In [ ]:
import pandas as pd
from datasets import load_dataset, \
    concatenate_datasets
from collections import Counter
from transformers import AutoTokenizer

In [ ]:
ds = load_dataset('NLBSE/nlbse26-code-comment-classification')

In [ ]:
ds["python_train"][1263]

In [ ]:
langs = ['java', 'python', 'pharo']
labels = {
    'java': ['summary', 'Ownership', 'Expand', 'usage', 'Pointer', 'deprecation', 'rational'],
    'python': ['Usage', 'Parameters', 'DevelopmentNotes', 'Expand', 'Summary'],
    'pharo': ['Keyimplementationpoints', 'Example', 'Responsibilities', 'Intent', 'Keymessages', 'Collaborators']
}
def split_list_into_columns(row, lang):
    values_list = row['labels']  # Replace 'values' with your actual column name
    dict = {}
    for key in labels[lang]:
        
        dict[key] = values_list[labels[lang].index(key)]

    return dict


java = concatenate_datasets([ds["java_train"]]).map(lambda row: split_list_into_columns(row, "java"))
python = concatenate_datasets([ds["python_train"]]).map(lambda row: split_list_into_columns(row, "python")).to_pandas()
pharo = concatenate_datasets([ds["pharo_train"]]).map(lambda row: split_list_into_columns(row, "pharo")).to_pandas()


In [ ]:

def balance_labels(ds,lang, target_ratio, tol=0.1, max_iter=10000):
    """
    Balance positive instances per label to reach a target ratio with a tolerance.
    
    Parameters:
    - d: dict of {label: {'positive': int, 'negative': int}}
    - target_ratio: desired positive/negative ratio
    - tol: allowed relative error (default 0.1 → 10%)
    - max_iter: maximum iterations to prevent infinite loops
    
    Returns:
    - dict with final positive and negative counts
    """
    labels = {
        'java': ['summary', 'Ownership', 'Expand', 'usage', 'Pointer', 'deprecation', 'rational'],
        'python': ['Usage', 'Parameters', 'DevelopmentNotes', 'Expand', 'Summary'],
        'pharo': ['Keyimplementationpoints', 'Example', 'Responsibilities', 'Classreferences', 'Intent', 'Keymessages', 'Collaborators']
    }

    d = {}
    for l in labels[lang]:
        #group = ds.groupby(l).count()
        group = Counter(ds[l])
        if l not in d.keys():
            d[l] = {"positive":group[1], 
                    "negative": group[0]}
            
    labels = list(d.keys())
    added_pos = {l: 0 for l in labels}
    changed = True
    iteration = 0
    
    while changed and iteration < max_iter:
        changed = False
        iteration += 1
        
        for l in labels:
            P = d[l]['positive'] + added_pos[l]
            N = d[l]['negative'] + sum(added_pos[other] for other in labels if other != l)
            current_ratio = P / N
            
            # Acceptable range considering tolerance
            if current_ratio < target_ratio * (1 - tol):
                added_pos[l] += 1
                changed = True
    
    if iteration == max_iter:
        print("Warning: maximum iterations reached. Result may not fully meet target ratio.")
    
    # Compute final counts
    result = {}
    resultL = []
    for l in labels:
        final_pos = d[l]['positive'] + added_pos[l]
        final_neg = d[l]['negative'] + sum(added_pos[other] for other in labels if other != l)
        result[l] = {'add': added_pos[l],'positive': final_pos, 'negative': final_neg}
        resultL.append(added_pos[l])
    
    return result,resultL

# Example usage
target_ratio = 0.15
balanced,balancedl = balance_labels(java,'java', target_ratio)
print(balancedl)
for l in labels['java']:
    print(l)
    print(balanced[l])
    print('*'*40)


In [ ]:
def print_labels(data, lang):
    for label in labels[lang]:
        group = Counter(data[label])
        print(group)
        print("label " + label)
        print("positive", group[1], ",negative", group[0], "--- %", (group[1]/(group[1]+group[0]))*100)
        print("number of augments | 10%", (group[0]-group[1])*0.1,
              "| 25%", (group[0]-group[1])*0.25,
              "| 50%", (group[0]-group[1])*0.5,
              "| 75%", (group[0]-group[1])*0.75,
              "| 100%", (group[0]-group[1]))
        print('*'*40)

In [ ]:
print_labels(java, "java")

In [ ]:
print_labels(python, "python")

In [ ]:
print_labels(pharo, "pharo")

In [ ]:
import pandas as pd
from tqdm import tqdm
import numpy as np

data = pd.concat([ds["java_train"].to_pandas(), ds["python_train"].to_pandas(), ds["pharo_train"].to_pandas()])
data["len_tokens"] = None
data["sumLabel"] = None

len(data)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

pbar = tqdm(total=len(data), desc="adding tokens length...")

for index, row in data.iterrows():
    input = row["combo"]
    tokens = tokenizer.encode(input)
    len_tokens = len(tokens)
    data.loc[index, "len_tokens"] = len_tokens
    data.loc[index, "sumLabel"] = sum(row["labels"])
    pbar.update(1)

pbar.close()

data

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
list_token_length = data['len_tokens']
min_value = np.min(list_token_length)
max_value = np.max(list_token_length)
average = np.mean(list_token_length)
median = np.median(list_token_length)

print(f"Min: {min_value}")
print(f"Max: {max_value}") # apparently, one single input is longer than 512 and is truncated - I dont see an issue there
print(f"Average: {average:.2f}")
print(f"Median: {median:.2f}")

sns.histplot(list_token_length, kde=False)
plt.title("Distribution Plot of token length")
plt.yscale('log') # Adding a log scale at the y-axis to give more visibility to the long inputs
plt.xlabel("Values")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

datasets = {
    "Java": ds["java_train"].to_pandas(),
    "Python": ds["python_train"].to_pandas(),
    "Pharo": ds["pharo_train"].to_pandas(),
}

for lang, df in datasets.items():
    # Compute token length and sum of labels
    df["len_tokens"] = df["combo"].apply(lambda x: len(tokenizer.encode(x)))
    df["sumLabel"] = df["labels"].apply(sum)
    df['hue']=df["labels"].apply(str)

    # Stats
    min_value = df["len_tokens"].min()
    max_value = df["len_tokens"].max()
    avg_value = df["len_tokens"].mean()
    median_value = df["len_tokens"].median()

    print(f"\n {lang} dataset:")
    print(f"Min: {min_value}")
    print(f"Max: {max_value}")
    print(f"Average: {avg_value:.2f}")
    print(f"Median: {median_value:.2f}")

    # Plot distribution colored by label count
    plt.figure(figsize=(8, 5))
    sns.histplot(data=df, x="len_tokens", hue="hue", bins=50, kde=False, multiple="stack",
    palette="tab10",
    legend=True)
    plt.title(f"Distribution of Token Lengths in {lang}", fontsize=14)
    plt.yscale('log')  # better visibility
    plt.xlabel("Token length", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    handles = [Patch(color=c, label=l) for l, c in zip(df["hue"].unique(), sns.color_palette("tab10"))]
    plt.legend(handles=handles, title="Labels", loc="upper right")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib.ticker import LogLocator
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

datasets = {
    "Java": ds["java_train"].to_pandas(),
    "Python": ds["python_train"].to_pandas(),
    "Pharo": ds["pharo_train"].to_pandas(),
}

fig, axes = plt.subplots(1, len(datasets), figsize=(20, 8), sharey=True)

for ax, (lang, df) in zip(axes, datasets.items()):
    # Compute token length and sum of labels
    df["len_tokens"] = df["combo"].apply(lambda x: len(tokenizer.encode(x)))
    df["sumLabel"] = df["labels"].apply(sum)
    df["hue"] = df["labels"].apply(str)

    # Stats
    min_value = df["len_tokens"].min()
    max_value = df["len_tokens"].max()
    avg_value = df["len_tokens"].mean()
    median_value = df["len_tokens"].median()


    # Plot on subplot
    sns.histplot(
        data=df,
        x="len_tokens",
        hue="hue",
        bins=50,
        kde=False,
        multiple="stack",
        palette="tab10",
        legend=False,
        ax=ax,
    )
    ax.set_title(f"{lang}", fontsize=14)
    ax.set_yscale("log")
    # Major ticks: powers of 10
    ax.yaxis.set_major_locator(LogLocator(base=10.0, subs=None, numticks=10))
    ax.grid(which="major", axis="y", linestyle="--", linewidth=0.65, alpha=0.65)

    # Minor ticks: between powers of 10
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=range(1, 10), numticks=10))
    ax.grid(which="minor", axis="y", linestyle="-", linewidth=1, alpha=0.8)
    ax.set_xlabel("Token length", fontsize=12)
    ax.set_ylabel("Frequency", fontsize=12 if ax == axes[0] else 0)
    ax.grid(True, linestyle="--", alpha=0.6)

# Shared legend
handles = [Patch(color=c, label=l) for l, c in zip(df["hue"].unique(), sns.color_palette("tab10"))]
fig.legend(handles=handles, title="Labels", loc="lower center",
    ncol=5,   # max per row
    frameon=False,)

fig.suptitle("Distribution of Token Lengths by Language", fontsize=16)
plt.tight_layout(rect=[0, 0.15, 0.9, 0.95])
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import LogLocator

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

datasets = {
    "Java": ds["java_train"].to_pandas(),
    "Python": ds["python_train"].to_pandas(),
    "Pharo": ds["pharo_train"].to_pandas(),
}

fig, axes = plt.subplots(1, len(datasets), figsize=(18, 5), sharey=True)

for ax, (lang, df) in zip(axes, datasets.items()):
    # Compute token length and sum of labels
    df["len_tokens"] = df["combo"].apply(lambda x: len(tokenizer.encode(x)))
    df["sumLabel"] = df["labels"].apply(sum)

    # Number of all-zero labels
    num_zero_labels = (df["sumLabel"] == 0).sum()
    print(f"{lang}: {num_zero_labels} observations with all-zero labels")

    # Frequency of each label count
    length_counts = df["sumLabel"].value_counts().sort_index()

    # Plot as bar chart
    sns.barplot(
        x=length_counts.index,
        y=length_counts.values,
        color="skyblue",
        ax=ax
    )

    ax.set_title(f"{lang}", fontsize=14)
    ax.set_xlabel("Labels number", fontsize=12)
    ax.set_ylabel("Frequency", fontsize=12 if ax == axes[0] else 0)
    ax.set_yscale("log")

    # Major ticks: powers of 10
    ax.yaxis.set_major_locator(LogLocator(base=10.0, subs=None, numticks=10))
    ax.grid(which="major", axis="y", linestyle="--", linewidth=0.65, alpha=0.65)

    # Minor ticks: between powers of 10
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=range(1, 10), numticks=10))
    ax.grid(which="minor", axis="y", linestyle="-", linewidth=1, alpha=0.8)

fig.suptitle("Distribution of Multi-Labels by Language", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
def generate_weights(weighted_loss, data):

    if weighted_loss == "frequency":
        labels = data["labels"]
        labels_array = np.array(labels)
        class_counts = labels_array.sum(axis=0)
        total_samples = len(labels)
        class_frequencies = (class_counts/total_samples)*100
        result = []

        for i, freq in enumerate(class_frequencies):
            result.append(round(1/freq, 2))

        result = np.array(result)

        return class_frequencies #* (1 / np.sum(result))

    if weighted_loss == "ranked":
        labels = data["labels"]
        labels_array = np.array(labels)
        class_counts = labels_array.sum(axis=0)
        total_samples = len(labels)
        class_frequencies = (class_counts/total_samples)*100

        sorted_values = sorted(range(len(class_frequencies)), key=lambda x: class_frequencies[x], reverse=True)
        sorted_frequencies = sorted(class_frequencies)

        mapped_output = [0] * len(class_frequencies)
        for i, idx in enumerate(sorted_values):
            mapped_output[idx] = sorted_frequencies[i]

        return mapped_output


    # NO WEIGHTED LOSS
    return [1] * len(data[0]["labels"])

In [ ]:
print(f"frequency: {generate_weights('frequency', ds['java_train'])}")
print(f"ranked: {generate_weights('ranked', ds['java_train'])}")
print(f"noweight: {generate_weights('no', ds['java_train'])}")